In [1]:
"""
================================================================================
PART 1 — DATA SAMPLING
ECG Image Digitization using Deep Learning
================================================================================
Purpose : Sample a representative 1 GB subset from the full 80 GB PhysioNet
          ECG Image Digitization dataset while preserving class distribution.

Input   : physionet-ecg-image-digitization/train/
Output  : Data_subset/

Strategy:
  1. Walk the source directory tree and collect every image file.
  2. Group files by their parent folder (acts as class / signal type label).
  3. Compute each group's proportional share of the 1 GB budget.
  4. Within each group, shuffle and select files until the quota is met.
  5. Copy selected files into Data_subset/, mirroring the original structure.

Key guarantees:
  - No bias  : Files are shuffled before selection (random.shuffle).
  - Balance  : Each folder/class gets its exact proportional byte budget.
  - Limit    : Total copied data never exceeds TARGET_SIZE_GB.
  - Portable : TARGET_SIZE_GB is adjustable via the constant below.
================================================================================
"""

import os
import shutil
import random
import argparse
from collections import defaultdict

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SOURCE_DIR      = "physionet-ecg-image-digitization/train"
DEST_DIR        = "Data_subset"
TARGET_SIZE_GB  = 1.0                          # change this to adjust subset size
VALID_EXTS      = (".png", ".jpg", ".jpeg", ".bmp", ".tiff")
RANDOM_SEED     = 42                           # for reproducibility

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def bytes_to_gb(b: int) -> float:
    return b / (1024 ** 3)

def gb_to_bytes(gb: float) -> int:
    return int(gb * (1024 ** 3))

def collect_files_by_class(source_dir: str) -> dict:
    """
    Walk source_dir and return a dict mapping each sub-folder name (class)
    to a list of (file_path, file_size_bytes) tuples.
    """
    class_files = defaultdict(list)
    for root, _, files in os.walk(source_dir):
        class_name = os.path.relpath(root, source_dir)   # relative folder = class label
        for fname in files:
            if fname.lower().endswith(VALID_EXTS):
                fpath = os.path.join(root, fname)
                fsize = os.path.getsize(fpath)
                class_files[class_name].append((fpath, fsize))
    return class_files

def compute_class_budgets(class_files: dict, total_budget_bytes: int) -> dict:
    """
    Allocate bytes proportionally to each class based on its share of the
    total dataset size, ensuring the subset mirrors the full distribution.
    """
    # Total dataset size
    total_bytes = sum(
        sz for files in class_files.values() for _, sz in files
    )

    budgets = {}
    for cls, files in class_files.items():
        cls_bytes  = sum(sz for _, sz in files)
        proportion = cls_bytes / total_bytes if total_bytes > 0 else 0
        budgets[cls] = int(proportion * total_budget_bytes)

    return budgets

def sample_files(class_files: dict, budgets: dict, seed: int) -> list:
    """
    For each class, randomly shuffle then greedily select files until the
    class byte budget is exhausted.  Returns a flat list of source paths.
    """
    rng = random.Random(seed)
    selected = []

    for cls, files in class_files.items():
        shuffled = files[:]
        rng.shuffle(shuffled)

        remaining = budgets.get(cls, 0)
        for fpath, fsize in shuffled:
            if remaining <= 0:
                break
            selected.append(fpath)
            remaining -= fsize

    return selected

def copy_subset(selected_paths: list, source_dir: str, dest_dir: str) -> None:
    """
    Copy selected files to dest_dir, preserving the original sub-folder
    structure relative to source_dir.
    """
    os.makedirs(dest_dir, exist_ok=True)
    total = len(selected_paths)

    for i, src_path in enumerate(selected_paths, start=1):
        rel_path = os.path.relpath(src_path, source_dir)
        dst_path = os.path.join(dest_dir, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(src_path, dst_path)

        if i % 500 == 0 or i == total:
            print(f"  Copied {i:>6}/{total}  →  {dst_path}")

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main(source_dir: str = SOURCE_DIR,
         dest_dir:   str = DEST_DIR,
         target_gb: float = TARGET_SIZE_GB):

    print("=" * 65)
    print("  ECG DATASET SAMPLING  —  1 GB Subset Creation")
    print("=" * 65)
    print(f"  Source      : {source_dir}")
    print(f"  Destination : {dest_dir}")
    print(f"  Target size : {target_gb} GB")
    print(f"  Random seed : {RANDOM_SEED}")
    print("=" * 65)

    # ── Step 1: Discover all files grouped by class ───────────────────────────
    print("\n[Step 1] Scanning source directory ...")
    class_files = collect_files_by_class(source_dir)

    if not class_files:
        raise FileNotFoundError(
            f"No image files found in '{source_dir}'.\n"
            f"Please check the path and ensure images are present."
        )

    total_source_bytes = sum(sz for files in class_files.values() for _, sz in files)
    total_source_files = sum(len(files) for files in class_files.values())

    print(f"  Classes (sub-folders) found : {len(class_files)}")
    print(f"  Total image files           : {total_source_files:,}")
    print(f"  Total dataset size          : {bytes_to_gb(total_source_bytes):.2f} GB")

    # ── Step 2: Compute per-class byte budgets ────────────────────────────────
    print("\n[Step 2] Computing proportional class budgets ...")
    target_bytes = gb_to_bytes(target_gb)
    budgets      = compute_class_budgets(class_files, target_bytes)

    for cls, budget in sorted(budgets.items()):
        n_files = len(class_files[cls])
        print(f"  {cls:<40}  budget: {bytes_to_gb(budget):.3f} GB  ({n_files} files)")

    # ── Step 3: Random proportional sampling ─────────────────────────────────
    print("\n[Step 3] Sampling files (seed={}) ...".format(RANDOM_SEED))
    selected = sample_files(class_files, budgets, RANDOM_SEED)

    selected_bytes = sum(os.path.getsize(p) for p in selected)
    print(f"  Files selected : {len(selected):,}")
    print(f"  Subset size    : {bytes_to_gb(selected_bytes):.3f} GB")

    # ── Step 4: Copy to destination ───────────────────────────────────────────
    print(f"\n[Step 4] Copying files to '{dest_dir}' ...")
    copy_subset(selected, source_dir, dest_dir)

    # ── Step 5: Verify and summarize ──────────────────────────────────────────
    print("\n[Step 5] Verification ...")
    dest_count = sum(
        1 for _, _, files in os.walk(dest_dir)
        for f in files if f.lower().endswith(VALID_EXTS)
    )
    print(f"  Files in destination : {dest_count:,}")
    print(f"  Subset size on disk  : {bytes_to_gb(selected_bytes):.3f} GB")

    print("\n" + "=" * 65)
    print("  SAMPLING COMPLETE  ✓")
    print(f"  Subset saved to: {os.path.abspath(dest_dir)}")
    print("=" * 65)


# ─────────────────────────────────────────────────────────────────────────────
# CLI ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Sample a representative subset from the PhysioNet ECG dataset."
    )
    parser.add_argument("--source",    default=SOURCE_DIR,     help="Source dataset directory")
    parser.add_argument("--dest",      default=DEST_DIR,       help="Output subset directory")
    parser.add_argument("--size_gb",   default=TARGET_SIZE_GB, type=float,
                        help="Target subset size in GB (default: 1.0)")
    args = parser.parse_args(args=[])
    
    main(source_dir=args.source, dest_dir=args.dest, target_gb=args.size_gb)


  ECG DATASET SAMPLING  —  1 GB Subset Creation
  Source      : physionet-ecg-image-digitization/train
  Destination : Data_subset
  Target size : 1.0 GB
  Random seed : 42

[Step 1] Scanning source directory ...
  Classes (sub-folders) found : 274
  Total image files           : 2,466
  Total dataset size          : 22.32 GB

[Step 2] Computing proportional class budgets ...
  1006427285                                budget: 0.004 GB  (9 files)
  1006867983                                budget: 0.004 GB  (9 files)
  1012423188                                budget: 0.004 GB  (9 files)
  10140238                                  budget: 0.004 GB  (9 files)
  1015663939                                budget: 0.004 GB  (9 files)
  102150619                                 budget: 0.004 GB  (9 files)
  1026034238                                budget: 0.004 GB  (9 files)
  1041099777                                budget: 0.004 GB  (9 files)
  104573050                                 b

In [2]:
"""
================================================================================
PARTS 2 & 3 — IMAGE PROCESSING PIPELINE + MULTI-IMAGE BATCH PROCESSING
ECG Image Digitization using Deep Learning
================================================================================
Input  : Data_subset/   (created by part1_data_sampling)
Output : Output/
         └── <image_name>/
             ├── 01_grayscale.png
             ├── 02_binary.png
             ├── 03_no_grid.png
             ├── 04_enhanced.png
             └── ecg_signal_plot.png
================================================================================
"""

import os
import cv2
import argparse
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INPUT_DIR      = "Data_subset"   # output of part1_data_sampling.py
OUTPUT_DIR     = "Output"
MAX_IMAGES     = 5               # demo limit (adjust as needed)

# Morphological parameters  ── do NOT change without re-tuning ──
GRID_KERNEL_H  = 40             # horizontal structuring element length (pixels)
GRID_KERNEL_V  = 40             # vertical structuring element length (pixels)
DILATE_ITER    = 2              # waveform dilation iterations
CLOSE_KERNEL   = 5              # waveform closing kernel side length

VALID_EXTS     = (".png", ".jpg", ".jpeg", ".bmp", ".tiff")


# ==============================================================================
# STEP 1 — LOAD & GRAYSCALE CONVERSION
# ==============================================================================
def load_and_grayscale(image_path: str):
    """
    Load a colour ECG image and convert to single-channel grayscale.

    Grayscale conversion removes irrelevant colour information and reduces
    the data volume before thresholding.  OpenCV uses the ITU-R BT.601
    luminance formula:  Y = 0.299R + 0.587G + 0.114B

    Args:
        image_path (str): Absolute or relative path to the input image.

    Returns:
        tuple[ndarray, ndarray]: (BGR image, grayscale image)

    Raises:
        FileNotFoundError: If the image cannot be read from disk.
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return img_bgr, img_gray


# ==============================================================================
# STEP 2 — BINARY THRESHOLDING  (Otsu's Method)
# ==============================================================================
def apply_threshold(gray: np.ndarray) -> np.ndarray:
    """
    Convert grayscale image to binary using Otsu's global thresholding.

    Otsu's algorithm automatically determines the optimal threshold T* by
    minimising the weighted intra-class variance between foreground and
    background pixel populations.  THRESH_BINARY_INV is used so that the
    dark ECG waveform becomes white (255) foreground.

    Args:
        gray (ndarray): Single-channel grayscale image.

    Returns:
        ndarray: Binary image — waveform pixels = 255, background = 0.
    """
    _, binary = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    return binary


# ==============================================================================
# STEP 3 — GRID REMOVAL  (Morphological Opening with Directional Kernels)
# ==============================================================================
def remove_grid(binary: np.ndarray,
                h_len:  int = GRID_KERNEL_H,
                v_len:  int = GRID_KERNEL_V) -> np.ndarray:
    """
    Isolate and remove the printed ECG grid using morphological operations.

    Approach:
      (a) Detect horizontal lines → opening with a wide flat kernel (h_len × 1).
      (b) Detect vertical lines   → opening with a tall thin kernel (1 × v_len).
      (c) Merge both line images into a combined grid mask.
      (d) Subtract grid mask from the binary image, leaving only the waveform.

    The kernel lengths are chosen to be longer than any single waveform feature
    but shorter than a full grid line, ensuring selectivity.

    Args:
        binary (ndarray): Binarized ECG image.
        h_len  (int)    : Horizontal kernel width  (pixels, default 40).
        v_len  (int)    : Vertical   kernel height (pixels, default 40).

    Returns:
        ndarray: Binary image with grid lines removed.
    """
    # (a) Detect horizontal grid lines
    h_kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (h_len, 1))
    h_lines   = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)

    # (b) Detect vertical grid lines
    v_kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (1, v_len))
    v_lines   = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel)

    # (c) Combine into full grid mask
    grid_mask = cv2.add(h_lines, v_lines)

    # (d) Remove grid from binary image
    no_grid   = cv2.subtract(binary, grid_mask)
    return no_grid


# ==============================================================================
# STEP 4 — WAVEFORM ENHANCEMENT  (Dilation + Morphological Closing)
# ==============================================================================
def enhance_waveform(no_grid:     np.ndarray,
                     dilate_iter: int = DILATE_ITER,
                     close_k:     int = CLOSE_KERNEL) -> np.ndarray:
    """
    Strengthen the isolated waveform pixels after grid removal.

    Two sequential morphological operations are applied:
      1. Dilation  — expands white waveform pixels, reconnecting small gaps
                     caused by grid subtraction or image noise.
      2. Closing   — fills internal holes in the waveform (dilation followed
                     by erosion with the same kernel), smoothing the signal
                     boundary without significantly altering its shape.

    Args:
        no_grid     (ndarray): Binary image after grid removal.
        dilate_iter (int)    : Number of dilation iterations (default 2).
        close_k     (int)    : Closing kernel side length in pixels (default 5).

    Returns:
        ndarray: Enhanced binary waveform image.
    """
    # Dilation — thicken and reconnect thin waveform strokes
    d_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    dilated  = cv2.dilate(no_grid, d_kernel, iterations=dilate_iter)

    # Closing — fill gaps and smooth boundaries
    c_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (close_k, close_k))
    enhanced = cv2.morphologyEx(dilated, cv2.MORPH_CLOSE, c_kernel)
    return enhanced


# ==============================================================================
# STEP 5 — SIGNAL COORDINATE EXTRACTION
# ==============================================================================
def extract_signal(enhanced: np.ndarray):
    """
    Convert the binary waveform image to a sequence of (x, y) signal coordinates.

    For every column x in the image, the set of active (white) pixel row indices
    is identified.  The waveform amplitude at x is taken as the vertical centroid
    of those rows.  The image y-axis is inverted so that larger y values
    correspond to higher signal amplitudes (physiological convention).

    Columns with no active pixels are skipped; this handles small gaps in the
    enhanced waveform without producing spurious zero-amplitude artefacts.

    Args:
        enhanced (ndarray): Enhanced binary waveform image (H × W).

    Returns:
        tuple[ndarray, ndarray]: (x_coords, y_coords) — both shape (N,).
    """
    h, w     = enhanced.shape
    x_coords = []
    y_coords = []

    for x in range(w):
        col         = enhanced[:, x]
        active_rows = np.where(col > 0)[0]
        if len(active_rows) > 0:
            y_centroid = int(np.mean(active_rows))
            x_coords.append(x)
            y_coords.append(h - y_centroid)   # invert y-axis

    return np.array(x_coords), np.array(y_coords)


# ==============================================================================
# STEP 6 — ECG SIGNAL PLOTTING
# ==============================================================================
def save_signal_plot(x:           np.ndarray,
                     y:           np.ndarray,
                     output_path: str,
                     title:       str = "Extracted ECG Signal") -> None:
    """
    Render and save the digitized ECG signal as a publication-quality PNG.

    Args:
        x           (ndarray): Column indices (proxy for time).
        y           (ndarray): Centroid values (proxy for amplitude).
        output_path (str)    : Full path for the output PNG file.
        title       (str)    : Figure title.
    """
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(x, y, color="#C0392B", linewidth=1.0, label="ECG Signal")
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Time (pixels)", fontsize=11)
    ax.set_ylabel("Amplitude (pixels)", fontsize=11)
    ax.legend(loc="upper right")
    ax.grid(True, linestyle="--", alpha=0.45, color="#888888")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# ==============================================================================
# CORE FUNCTION — process_ecg()
# ==============================================================================
def process_ecg(image_path: str, filename: str) -> None:
    """
    Execute the complete 6-stage ECG digitization pipeline on one image and
    save all intermediate outputs to Output/<filename_without_extension>/.

    Saved files:
        01_grayscale.png   — Stage 1 output
        02_binary.png      — Stage 2 output
        03_no_grid.png     — Stage 3 output
        04_enhanced.png    — Stage 4 output
        ecg_signal_plot.png — Stage 5+6 combined output

    Args:
        image_path (str): Full path to the source ECG image.
        filename   (str): Image filename including extension (e.g. "ecg01.png").
    """
    # Create per-image output subfolder
    base_name  = os.path.splitext(filename)[0]
    out_folder = os.path.join(OUTPUT_DIR, base_name)
    os.makedirs(out_folder, exist_ok=True)

    print(f"\n{'─' * 60}")
    print(f"  File       : {filename}")
    print(f"  Output dir : {out_folder}")
    print(f"{'─' * 60}")

    # ── Stage 1: Load & Grayscale ─────────────────────────────────────────────
    img_bgr, img_gray = load_and_grayscale(image_path)
    cv2.imwrite(os.path.join(out_folder, "01_grayscale.png"), img_gray)
    print(f"  [1/5] 01_grayscale.png     — shape {img_gray.shape}")

    # ── Stage 2: Otsu Thresholding ────────────────────────────────────────────
    binary = apply_threshold(img_gray)
    cv2.imwrite(os.path.join(out_folder, "02_binary.png"), binary)
    print(f"  [2/5] 02_binary.png        — Otsu thresholding applied")

    # ── Stage 3: Grid Removal ────────────────────────────────────────────────
    no_grid = remove_grid(binary)
    cv2.imwrite(os.path.join(out_folder, "03_no_grid.png"), no_grid)
    print(f"  [3/5] 03_no_grid.png       — H-kernel {GRID_KERNEL_H}px, V-kernel {GRID_KERNEL_V}px")

    # ── Stage 4: Waveform Enhancement ────────────────────────────────────────
    enhanced = enhance_waveform(no_grid)
    cv2.imwrite(os.path.join(out_folder, "04_enhanced.png"), enhanced)
    print(f"  [4/5] 04_enhanced.png      — dilation ×{DILATE_ITER}, closing {CLOSE_KERNEL}×{CLOSE_KERNEL}")

    # ── Stage 5+6: Extract Signal & Plot ─────────────────────────────────────
    x, y      = extract_signal(enhanced)
    plot_path = os.path.join(out_folder, "ecg_signal_plot.png")
    save_signal_plot(x, y, plot_path, title=f"ECG Signal — {base_name}")
    print(f"  [5/5] ecg_signal_plot.png  — {len(x):,} data points")

    print(f"  ✓ Done")


# ==============================================================================
# BATCH RUNNER
# ==============================================================================
def run_batch(input_dir: str = INPUT_DIR, max_images: int = MAX_IMAGES) -> None:
    """
    Scan input_dir for ECG images and process the first max_images through
    the complete pipeline, saving results in per-image subfolders.

    Args:
        input_dir  (str): Root directory containing ECG images (may be nested).
        max_images (int): Maximum number of images to process.
    """
    if not os.path.isdir(input_dir):
        raise NotADirectoryError(
            f"Input directory not found: '{input_dir}'\n"
            f"Run part1_data_sampling.py first to create Data_subset/."
        )

    # Collect all valid images recursively, sorted for reproducibility
    all_images = []
    for root, _, files in os.walk(input_dir):
        for fname in sorted(files):
            if fname.lower().endswith(VALID_EXTS):
                all_images.append((os.path.join(root, fname), fname))

    if not all_images:
        raise FileNotFoundError(
            f"No image files found in '{input_dir}'.\n"
            f"Supported formats: {VALID_EXTS}"
        )

    selected = all_images[:max_images]
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── Header ───────────────────────────────────────────────────────────────
    print("=" * 60)
    print("  ECG DIGITIZATION — BATCH PROCESSING PIPELINE")
    print("=" * 60)
    print(f"  Input  folder : {input_dir}")
    print(f"  Output folder : {OUTPUT_DIR}")
    print(f"  Images found  : {len(all_images):,}")
    print(f"  Processing    : first {len(selected)}")
    print("=" * 60)

    success, failed = 0, 0
    for idx, (img_path, fname) in enumerate(selected, start=1):
        print(f"\n[{idx}/{len(selected)}]")
        try:
            process_ecg(img_path, fname)
            success += 1
        except Exception as err:
            print(f"  ✗ ERROR — {fname}: {err}")
            failed += 1

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"  BATCH COMPLETE")
    print(f"  Successful : {success}")
    print(f"  Failed     : {failed}")
    print(f"  Results    : {os.path.abspath(OUTPUT_DIR)}/")
    print("=" * 60)


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="ECG image digitization — classical pipeline (Parts 2 & 3)."
    )
    parser.add_argument("--src", default=INPUT_DIR,  help="Input image folder")
    parser.add_argument("--n",   default=MAX_IMAGES, type=int,
                        help="Maximum number of images to process (default: 5)")
    args = parser.parse_args(args=[])

    run_batch(input_dir=args.src, max_images=args.n)


  ECG DIGITIZATION — BATCH PROCESSING PIPELINE
  Input  folder : Data_subset
  Output folder : Output
  Images found  : 341
  Processing    : first 5

[1/5]

────────────────────────────────────────────────────────────
  File       : 1006427285-0005.png
  Output dir : Output\1006427285-0005
────────────────────────────────────────────────────────────
  [1/5] 01_grayscale.png     — shape (3024, 4032)
  [2/5] 02_binary.png        — Otsu thresholding applied
  [3/5] 03_no_grid.png       — H-kernel 40px, V-kernel 40px
  [4/5] 04_enhanced.png      — dilation ×2, closing 5×5
  [5/5] ecg_signal_plot.png  — 3,237 data points
  ✓ Done

[2/5]

────────────────────────────────────────────────────────────
  File       : 1006867983-0011.png
  Output dir : Output\1006867983-0011
────────────────────────────────────────────────────────────
  [1/5] 01_grayscale.png     — shape (1652, 2136)
  [2/5] 02_binary.png        — Otsu thresholding applied
  [3/5] 03_no_grid.png       — H-kernel 40px, V-kernel 4

In [3]:

"""
================================================================================
PART 4 — DEEP LEARNING MODEL  (FIXED for PhysioNet folder structure)
ECG Image Digitization using Deep Learning
================================================================================
Architecture : U-Net (Ronneberger et al., 2015)
Framework    : PyTorch

FIXED: Now correctly handles the PhysioNet nested folder structure:
    Data_subset/
    ├── 1006427285/          ← patient/record ID folders
    │   ├── image001.png
    │   └── image002.png
    ├── 1012423188/
    │   └── ...
    └── ...

Since the dataset has NO pre-made masks, this script auto-generates
binary masks using the classical morphological pipeline (same logic as
part2_3_pipeline.py) and caches them to Data_subset_masks/ so they are
only computed once.

Usage:
    python part4_unet.py                     # generate masks + train
    python part4_unet.py --skip_masks        # skip mask generation (already done)
    python part4_unet.py --epochs 30 --bs 4  # custom hyperparameters
    python part4_unet.py --infer img.png     # single-image inference
================================================================================
"""

import os
import cv2
import argparse
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ── Paths ────────────────────────────────────────────────────────────────
    "data_root":    "Data_subset",          # root with patient-ID subfolders
    "mask_root":    "Data_subset_masks",    # auto-generated masks saved here
    "model_dir":    "Output/models",        # checkpoints + curves saved here

    # ── Model ────────────────────────────────────────────────────────────────
    "img_size":     256,
    "in_channels":  1,
    "out_channels": 1,

    # ── Training ─────────────────────────────────────────────────────────────
    "batch_size":   2,
    "num_epochs":   8,
    "lr":           1e-4,
    "lr_patience":  5,
    "lr_factor":    0.5,
    "val_split":    0.2,
    "threshold":    0.5,
    "seed":         42,

    # ── Classical pipeline params (for mask generation) ───────────────────
    "grid_h":       40,
    "grid_v":       40,
    "dilate_iter":  2,
    "close_k":      5,
}

VALID_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tiff")
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"


# ==============================================================================
# STEP 0 — AUTO MASK GENERATION
# (Runs the classical pipeline to create proxy ground-truth masks)
# ==============================================================================
def _make_mask(img_path: str, cfg: dict) -> np.ndarray:
    """
    Generate a binary waveform mask for one image using the classical pipeline.
    This serves as a proxy ground-truth mask for supervised U-Net training.
    """
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(img_path)

    # Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Otsu threshold
    _, binary = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Grid removal
    h_ker     = cv2.getStructuringElement(cv2.MORPH_RECT, (cfg["grid_h"], 1))
    h_lines   = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_ker)
    v_ker     = cv2.getStructuringElement(cv2.MORPH_RECT, (1, cfg["grid_v"]))
    v_lines   = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_ker)
    no_grid   = cv2.subtract(binary, cv2.add(h_lines, v_lines))

    # Waveform enhancement
    d_ker     = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    dilated   = cv2.dilate(no_grid, d_ker, iterations=cfg["dilate_iter"])
    c_ker     = cv2.getStructuringElement(cv2.MORPH_RECT,
                                          (cfg["close_k"], cfg["close_k"]))
    mask      = cv2.morphologyEx(dilated, cv2.MORPH_CLOSE, c_ker)
    return mask   # uint8, values 0 or 255


def generate_masks(cfg: dict) -> None:
    """
    Walk Data_subset/ recursively, find all images, generate a binary mask
    for each using the classical pipeline, and save to Data_subset_masks/
    mirroring the original folder structure.

    This is only run once; existing masks are skipped.
    """
    data_root = cfg["data_root"]
    mask_root = cfg["mask_root"]
    os.makedirs(mask_root, exist_ok=True)

    # Collect all image paths
    all_imgs = []
    for root, _, files in os.walk(data_root):
        for f in sorted(files):
            if f.lower().endswith(VALID_EXTS):
                all_imgs.append(os.path.join(root, f))

    if not all_imgs:
        raise FileNotFoundError(
            f"No images found in '{data_root}'.\n"
            f"Ensure Data_subset/ contains patient-ID subfolders with images."
        )

    print(f"\n[Mask Generation] Found {len(all_imgs):,} images in '{data_root}'")
    print(f"[Mask Generation] Saving masks to '{mask_root}' ...")

    generated, skipped, failed = 0, 0, 0

    for i, img_path in enumerate(all_imgs, 1):
        # Mirror path under mask_root
        rel       = os.path.relpath(img_path, data_root)
        mask_path = os.path.join(mask_root, rel)

        # Skip if mask already exists
        if os.path.exists(mask_path):
            skipped += 1
            continue

        os.makedirs(os.path.dirname(mask_path), exist_ok=True)

        try:
            mask = _make_mask(img_path, cfg)
            cv2.imwrite(mask_path, mask)
            generated += 1
        except Exception as e:
            print(f"  WARN: Failed on {img_path}: {e}")
            failed += 1

        if i % 200 == 0 or i == len(all_imgs):
            print(f"  Progress: {i:>6}/{len(all_imgs)}  "
                  f"(generated={generated}, skipped={skipped}, failed={failed})")

    print(f"\n  Mask generation complete ✓")
    print(f"  Generated : {generated:,}")
    print(f"  Skipped   : {skipped:,}  (already existed)")
    print(f"  Failed    : {failed:,}")


# ==============================================================================
# DATASET — Handles nested PhysioNet folder structure
# ==============================================================================
def _collect_paired_paths(data_root: str, mask_root: str) -> list:
    """
    Recursively scan data_root for images and pair each with its corresponding
    mask in mask_root (same relative path).  Only pairs where both image AND
    mask exist are returned.

    Returns:
        list of (image_path, mask_path) tuples
    """
    pairs = []
    for root, _, files in os.walk(data_root):
        for fname in sorted(files):
            if not fname.lower().endswith(VALID_EXTS):
                continue
            img_path  = os.path.join(root, fname)
            rel       = os.path.relpath(img_path, data_root)
            mask_path = os.path.join(mask_root, rel)
            if os.path.exists(mask_path):
                pairs.append((img_path, mask_path))

    return pairs


class ECGSegmentationDataset(Dataset):
    """
    ECG image + binary mask dataset.

    Automatically discovers all (image, mask) pairs by recursively scanning
    data_root and mask_root.  Works with ANY nesting depth — patient-ID
    subfolders, class subfolders, flat structure, or mixed.

    Args:
        data_root (str): Root directory containing ECG images (nested OK).
        mask_root (str): Root directory containing corresponding masks.
        img_size  (int): Both dimensions are resized to this value.
    """

    def __init__(self, data_root: str, mask_root: str, img_size: int = 512):
        self.pairs = _collect_paired_paths(data_root, mask_root)

        if len(self.pairs) == 0:
            raise RuntimeError(
                f"No (image, mask) pairs found.\n"
                f"  data_root = {data_root}\n"
                f"  mask_root = {mask_root}\n"
                f"Run without --skip_masks to auto-generate masks first."
            )

        self.img_size  = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),       # → float32 in [0, 1]
        ])

        print(f"  Dataset: {len(self.pairs):,} valid (image, mask) pairs found")

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        img_path, mask_path = self.pairs[idx]

        image = Image.open(img_path).convert("L")    # grayscale
        mask  = Image.open(mask_path).convert("L")   # grayscale

        image = self.transform(image)                 # (1, H, W)
        mask  = self.transform(mask)                  # (1, H, W)
        mask  = (mask > 0.5).float()                  # binarize

        return image, mask

# ==============================================================================
# U-NET MODEL
# ==============================================================================
class _DoubleConv(nn.Module):
    """Conv(3×3)→BN→ReLU→Conv(3×3)→BN→ReLU"""
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class _EncoderBlock(nn.Module):
    """DoubleConv + MaxPool(2×2) → returns (skip, pooled)"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = _DoubleConv(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        s = self.conv(x)
        return s, self.pool(s)


class _DecoderBlock(nn.Module):
    """ConvTranspose(2×2) → cat(skip) → DoubleConv"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = _DoubleConv(in_ch, out_ch)   # in_ch = out_ch*2 after cat
    def forward(self, x, skip):
        x = self.up(x)
        # Pad if spatial sizes differ (odd input dimensions)
        if x.shape != skip.shape:
            x = nn.functional.interpolate(x, size=skip.shape[2:])
        return self.conv(torch.cat([x, skip], dim=1))


class UNet(nn.Module):
    """
    U-Net for binary ECG waveform segmentation.

    Input  : (B, 1, H, W)   — grayscale ECG image
    Output : (B, 1, H, W)   — waveform probability map [0, 1]

    Channel progression:
        Encoder : 1 → 64 → 128 → 256 → 512
        Bottleneck : 512 → 1024
        Decoder : 1024 → 512 → 256 → 128 → 64 → 1 (sigmoid)
    """
    def __init__(self, in_ch: int = 1, out_ch: int = 1):
        super().__init__()
        self.enc1 = _EncoderBlock(in_ch,  64)
        self.enc2 = _EncoderBlock(64,    128)
        self.enc3 = _EncoderBlock(128,   256)
        self.enc4 = _EncoderBlock(256,   512)
        self.btn  = _DoubleConv(512, 1024)
        self.dec4 = _DecoderBlock(1024, 512)
        self.dec3 = _DecoderBlock(512,  256)
        self.dec2 = _DecoderBlock(256,  128)
        self.dec1 = _DecoderBlock(128,   64)
        self.out  = nn.Sequential(
            nn.Conv2d(64, out_ch, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        s1, p1 = self.enc1(x)
        s2, p2 = self.enc2(p1)
        s3, p3 = self.enc3(p2)
        s4, p4 = self.enc4(p3)
        b      = self.btn(p4)
        d      = self.dec4(b,  s4)
        d      = self.dec3(d,  s3)
        d      = self.dec2(d,  s2)
        d      = self.dec1(d,  s1)
        return self.out(d)


# ==============================================================================
# LOSS FUNCTION — BCE + Dice
# ==============================================================================
class BCEDiceLoss(nn.Module):
    """
    L = BCE(pred, target) + (1 − DiceCoeff(pred, target))
    Dice loss handles class imbalance (thin waveform vs large background).
    """
    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.bce    = nn.BCELoss()
        self.smooth = smooth

    def dice(self, p, t):
        inter = (p * t).sum()
        return 1 - (2 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)

    def forward(self, pred, target):
        return self.bce(pred, target) + self.dice(pred, target)


# ==============================================================================
# METRICS
# ==============================================================================
def dice_score(pred, target, threshold=0.5, smooth=1e-6):
    p   = (pred > threshold).float()
    inter = (p * target).sum(dim=(1,2,3))
    d   = (2*inter + smooth) / (p.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + smooth)
    return d.mean().item()


# ==============================================================================
# TRAINING LOOP
# ==============================================================================
def train(cfg: dict) -> dict:
    """Full training loop with checkpointing and curve plotting."""
    torch.manual_seed(cfg["seed"])
    os.makedirs(cfg["model_dir"], exist_ok=True)

    # ── Dataset ───────────────────────────────────────────────────────────────
    print("\n[Training] Building dataset ...")
    dataset  = ECGSegmentationDataset(
        cfg["data_root"], cfg["mask_root"], cfg["img_size"]
    )
    dataset.pairs = dataset.pairs[:200]
    val_n    = int(len(dataset) * cfg["val_split"])
    trn_n    = len(dataset) - val_n
    trn_ds, val_ds = random_split(
        dataset, [trn_n, val_n],
        generator=torch.Generator().manual_seed(cfg["seed"])
    )
    use_cuda = torch.cuda.is_available()

    trn_dl = DataLoader(trn_ds, batch_size=cfg["batch_size"],
                        shuffle=True, num_workers=0, pin_memory=False)
    val_dl = DataLoader(val_ds, batch_size=cfg["batch_size"],
                        shuffle=False, num_workers=0, pin_memory=False)
    print(f"  Train samples : {trn_n:,}")
    print(f"  Val   samples : {val_n:,}")

    # ── Model / Loss / Optimiser ──────────────────────────────────────────────
    model     = UNet(cfg["in_channels"], cfg["out_channels"]).to(DEVICE)
    criterion = BCEDiceLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=cfg["lr_patience"], factor=cfg["lr_factor"],
    )

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters : {n_params:,}")
    print(f"  Device               : {DEVICE.upper()}")

    # ── Train ─────────────────────────────────────────────────────────────────
    history   = {"trn_loss": [], "val_loss": [], "trn_dice": [], "val_dice": []}
    best_loss = float("inf")
    best_path = os.path.join(cfg["model_dir"], "unet_ecg_best.pth")

    print(f"\n{'='*70}")
    print(f"  {'Epoch':>5}  {'TrnLoss':>9}  {'ValLoss':>9}  "
          f"{'TrnDice':>9}  {'ValDice':>9}  {'LR':>10}")
    print("=" * 70)

    for epoch in range(1, cfg["num_epochs"] + 1):

        # Train pass
        model.train()
        tl, td = 0.0, 0.0
        for imgs, masks in trn_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            pred  = model(imgs)
            loss  = criterion(pred, masks)
            loss.backward()
            optimizer.step()
            tl += loss.item()
            td += dice_score(pred, masks, cfg["threshold"])
        tl /= len(trn_dl); td /= len(trn_dl)

        # Val pass
        model.eval()
        vl, vd = 0.0, 0.0
        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                pred = model(imgs)
                vl  += criterion(pred, masks).item()
                vd  += dice_score(pred, masks, cfg["threshold"])
        vl /= len(val_dl); vd /= len(val_dl)

        scheduler.step(vl)
        lr = optimizer.param_groups[0]["lr"]

        history["trn_loss"].append(tl)
        history["val_loss"].append(vl)
        history["trn_dice"].append(td)
        history["val_dice"].append(vd)

        print(f"  {epoch:>5}  {tl:>9.4f}  {vl:>9.4f}  "
              f"{td:>9.4f}  {vd:>9.4f}  {lr:>10.2e}")

        # Save best checkpoint
        if vl < best_loss:
            best_loss = vl
            torch.save({
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_loss":    vl,
                "val_dice":    vd,
                "cfg":         cfg,
            }, best_path)

    print("=" * 70)
    print(f"\n  Best model → {best_path}  (val_loss={best_loss:.4f})")

    _plot_curves(history, cfg["model_dir"])
    return history


def _plot_curves(history, model_dir):
    ep = range(1, len(history["trn_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))

    ax[0].plot(ep, history["trn_loss"], label="Train", color="#2980B9", lw=2)
    ax[0].plot(ep, history["val_loss"],  label="Val",   color="#E74C3C", lw=2)
    ax[0].set_title("BCE + Dice Loss", fontweight="bold")
    ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
    ax[0].legend(); ax[0].grid(alpha=0.35)

    ax[1].plot(ep, history["trn_dice"], label="Train", color="#27AE60", lw=2)
    ax[1].plot(ep, history["val_dice"],  label="Val",   color="#F39C12", lw=2)
    ax[1].set_title("Dice Coefficient", fontweight="bold")
    ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Dice")
    ax[1].legend(); ax[1].grid(alpha=0.35)

    fig.tight_layout()
    out = os.path.join(model_dir, "training_curves.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Training curves → {out}")


# ==============================================================================
# INFERENCE
# ==============================================================================
def predict(model_path: str, image_path: str, cfg: dict) -> np.ndarray:
    """Load trained checkpoint and produce binary mask for one image."""
    ckpt   = torch.load(model_path, map_location=DEVICE)
    c      = ckpt.get("cfg", cfg)
    model  = UNet(c["in_channels"], c["out_channels"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    tf  = transforms.Compose([
        transforms.Resize((c["img_size"], c["img_size"])),
        transforms.ToTensor(),
    ])
    img    = Image.open(image_path).convert("L")
    tensor = tf(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        prob = model(tensor).squeeze().cpu().numpy()

    return (prob > c["threshold"]).astype(np.uint8) * 255


# ==============================================================================
# ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="U-Net ECG segmentation — fixed for PhysioNet folder structure."
    )
    parser.add_argument("--epochs",      type=int,   default=CFG["num_epochs"])
    parser.add_argument("--bs",          type=int,   default=CFG["batch_size"])
    parser.add_argument("--lr",          type=float, default=CFG["lr"])
    parser.add_argument("--data_root",   type=str,   default=CFG["data_root"])
    parser.add_argument("--mask_root",   type=str,   default=CFG["mask_root"])
    parser.add_argument("--skip_masks",  action="store_true",
                        help="Skip mask generation (use if already generated)")
    parser.add_argument("--infer",       type=str,   default=None,
                        help="Path to a single ECG image for inference only")
    args = parser.parse_args(args=[])

    CFG["num_epochs"] = args.epochs
    CFG["batch_size"] = args.bs
    CFG["lr"]         = args.lr
    CFG["data_root"]  = args.data_root
    CFG["mask_root"]  = args.mask_root

    print("=" * 70)
    print("  U-NET ECG SEGMENTATION  —  PhysioNet Nested Folder Structure")
    print("=" * 70)
    print(f"  data_root  : {CFG['data_root']}")
    print(f"  mask_root  : {CFG['mask_root']}")
    print(f"  model_dir  : {CFG['model_dir']}")
    print(f"  img_size   : {CFG['img_size']} × {CFG['img_size']}")
    print(f"  epochs     : {CFG['num_epochs']}")
    print(f"  batch_size : {CFG['batch_size']}")
    print(f"  lr         : {CFG['lr']}")
    print(f"  device     : {DEVICE.upper()}")
    print("=" * 70)

    if args.infer:
        # ── Inference only ────────────────────────────────────────────────────
        model_path = os.path.join(CFG["model_dir"], "unet_ecg_best.pth")
        if not os.path.exists(model_path):
            raise FileNotFoundError(
                f"No trained model at {model_path}. Train first."
            )
        mask     = predict(model_path, args.infer, CFG)
        out_path = args.infer.rsplit(".", 1)[0] + "_mask.png"
        cv2.imwrite(out_path, mask)
        print(f"  Mask saved → {out_path}")

    else:
        # ── Step 1: Generate masks ────────────────────────────────────────────
        if not args.skip_masks:
            print("\n[Step 1/2] Generating binary masks from classical pipeline ...")
            generate_masks(CFG)
        else:
            print("\n[Step 1/2] Skipping mask generation (--skip_masks set)")

        # ── Step 2: Train U-Net ───────────────────────────────────────────────
        print("\n[Step 2/2] Training U-Net ...")
        history = train(CFG)

        print(f"\n{'='*70}")
        print(f"  TRAINING COMPLETE")
        print(f"  Final Train Loss : {history['trn_loss'][-1]:.4f}")
        print(f"  Final Val Loss   : {history['val_loss'][-1]:.4f}")
        print(f"  Final Train Dice : {history['trn_dice'][-1]:.4f}")
        print(f"  Final Val Dice   : {history['val_dice'][-1]:.4f}")
        print(f"{'='*70}")

  U-NET ECG SEGMENTATION  —  PhysioNet Nested Folder Structure
  data_root  : Data_subset
  mask_root  : Data_subset_masks
  model_dir  : Output/models
  img_size   : 256 × 256
  epochs     : 8
  batch_size : 2
  lr         : 0.0001
  device     : CPU

[Step 1/2] Generating binary masks from classical pipeline ...

[Mask Generation] Found 341 images in 'Data_subset'
[Mask Generation] Saving masks to 'Data_subset_masks' ...
  Progress:    200/341  (generated=200, skipped=0, failed=0)
  Progress:    341/341  (generated=341, skipped=0, failed=0)

  Mask generation complete ✓
  Generated : 341
  Skipped   : 0  (already existed)
  Failed    : 0

[Step 2/2] Training U-Net ...

[Training] Building dataset ...
  Dataset: 341 valid (image, mask) pairs found
  Train samples : 160
  Val   samples : 40
  Trainable parameters : 31,036,481
  Device               : CPU

  Epoch    TrnLoss    ValLoss    TrnDice    ValDice          LR
      1     1.3213     1.0523     0.3439     0.4638    1.00e-04
    